# WavTRGAN: corrected evaluation

Run all cells in order (GPU and Internet on). Edit the paths in the **Configuration** cell first.

- Runs the trained model on the fixed test pairs (no random crop, flip or shuffle) and saves `input/`, `gt/` and `wavtrgan/` as 256×256 PNGs with the same file names.
- Scores PSNR and SSIM on float RGB images, PSNR/SSIM on the corrupted region only, PSNR on the known region, LPIPS (AlexNet) and FID.
- Writes `/kaggle/working/metrics.csv`. Baseline outputs with the same file names can be added in `BASELINES`.

In [ ]:
!pip install -q pytorch_wavelets lpips torch-fidelity

## Network (needed so `torch.load` can rebuild the saved model)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # used inside the network code

import math


from collections import OrderedDict
from pytorch_wavelets import DWTForward, DWTInverse
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F

import copy
#import logging
import math

from os.path import join as pjoin

import torch
import torch.nn as nn
import numpy as np

from torch.nn import CrossEntropyLoss, Dropout, Softmax, Linear, Conv2d, LayerNorm
from torch.nn.modules.utils import _pair
from scipy import ndimage

######################Wavelets_function############################
##################################################################



###########################################
#########Resnet Network###################
#########################################

class StdConv2d(nn.Conv2d):

    def forward(self, x):
        w = self.weight
        v, m = torch.var_mean(w, dim=[1, 2, 3], keepdim=True, unbiased=False)
        w = (w - m) / torch.sqrt(v + 1e-5)
        return F.conv2d(x, w, self.bias, self.stride, self.padding,
                        self.dilation, self.groups)


def conv3x3(cin, cout, stride=1, groups=1, bias=False):
    return StdConv2d(cin, cout, kernel_size=3, stride=stride,
                     padding=1, bias=bias, groups=groups)


def conv1x1(cin, cout, stride=1, bias=False):
    return StdConv2d(cin, cout, kernel_size=1, stride=stride,
                     padding=0, bias=bias)


class PreActBottleneck(nn.Module):
    """Pre-activation (v2) bottleneck block.
    """

    def __init__(self, cin, cout=None, cmid=None, stride=1):
        super().__init__()
        cout = cout or cin
        cmid = cmid or cout//4

        self.gn1 = nn.GroupNorm(32, cmid, eps=1e-6)
        self.conv1 = conv1x1(cin, cmid, bias=False)
        self.gn2 = nn.GroupNorm(32, cmid, eps=1e-6)
        self.conv2 = conv3x3(cmid, cmid, stride, bias=False)  # Original code has it on conv1!!
        self.gn3 = nn.GroupNorm(32, cout, eps=1e-6)
        self.conv3 = conv1x1(cmid, cout, bias=False)
        self.relu = nn.ReLU(inplace=True)

        if (stride != 1 or cin != cout):
            # Projection also with pre-activation according to paper.
            self.downsample = conv1x1(cin, cout, stride, bias=False)
            self.gn_proj = nn.GroupNorm(cout, cout)

    def forward(self, x):

        # Residual branch
        residual = x
        if hasattr(self, 'downsample'):
            residual = self.downsample(x)
            residual = self.gn_proj(residual)

        # Unit's branch
        y = self.relu(self.gn1(self.conv1(x)))
        y = self.relu(self.gn2(self.conv2(y)))
        y = self.gn3(self.conv3(y))

        y = self.relu(residual + y)
        return y

class ResNetV2(nn.Module):
    """Implementation of Pre-activation (v2) ResNet mode."""

    def __init__(self, block_units, width_factor):
        super().__init__()
        width = int(64 * width_factor)
        self.width = width

        self.root = nn.Sequential(OrderedDict([
            ('conv', StdConv2d(3, width, kernel_size=7, stride=2, bias=False, padding=3)),
            ('gn',nn.GroupNorm(32, width, eps=1e-6)),
            ('relu', nn.ReLU(inplace=True)),
            #('pool', nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        ]))

        self.body = nn.Sequential(OrderedDict([
            ('block1', nn.Sequential(OrderedDict(
                [('unit1', PreActBottleneck(cin=width, cout=width*4, cmid=width))] +
                [(f'unit{i:d}', PreActBottleneck(cin=width*4, cout=width*4, cmid=width)) for i in range(2, block_units[0] + 1)],
                ))),
            ('block2', nn.Sequential(OrderedDict(
                [('unit1', PreActBottleneck(cin=width*4, cout=width*8, cmid=width*2, stride=2))] +
                [(f'unit{i:d}', PreActBottleneck(cin=width*8, cout=width*8, cmid=width*2)) for i in range(2, block_units[1] + 1)],
                ))),
            ('block3', nn.Sequential(OrderedDict(
                [('unit1', PreActBottleneck(cin=width*8, cout=width*16, cmid=width*4, stride=2))] +
                [(f'unit{i:d}', PreActBottleneck(cin=width*16, cout=width*16, cmid=width*4)) for i in range(2, block_units[2] + 1)],
                ))),
            
        ]))

    def forward(self, x):
        features = []
        b, c, in_size, _ = x.size()
        x = self.root(x)
        features.append(x)
        x = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)(x)
        for i in range(len(self.body)-1):
            x = self.body[i](x)
            right_size = int(in_size / 4 / (i+1))
            if x.size()[2] != right_size:
                pad = right_size - x.size()[2]
                assert pad < 3 and pad > 0, "x {} should {}".format(x.size(), right_size)
                feat = torch.zeros((b, x.size()[1], right_size, right_size), device=x.device)
                feat[:, :, 0:x.size()[2], 0:x.size()[3]] = x[:]
            else:
                feat = x
            features.append(feat)
        x = self.body[-1](x)
        return x, features[::-1]
    
#####################################################
############Vision Transformer Net###################
####################################################
################ Wavlet_Suery#######################


class SSL(nn.Module):
    def __init__(self, channels):
        super(SSL, self).__init__()

        # Convolutional layers for processing wavelet components
        self.conv_approx = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.conv_horiz = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.conv_vert = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.conv_diag = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        # Apply DWT to decompose the input image
        dwt = DWTForward(J=1, mode='zero', wave='db3').to(x.device)
        yl, yh = dwt(x)

        # Extract detail coefficients
        yh_out = yh[0]
        ylh = yh_out[:, :, 0, :, :]  # Horizontal details
        yhl = yh_out[:, :, 1, :, :]  # Vertical details
        yhh = yh_out[:, :, 2, :, :]  # Diagonal details

        # Process each wavelet component with CNNs
        approx_features = self.conv_approx(yl)
        horiz_features = self.conv_horiz(ylh)
        vert_features = self.conv_vert(yhl)
        diag_features = self.conv_diag(yhh)

        # Reconstruct each component using the inverse wavelet transform
        ifm = DWTInverse(wave='db3', mode='zero').to(x.device)

        # Reconstruct horizontal component
        rec_horiz = ifm((approx_features, [torch.stack((horiz_features, torch.zeros_like(horiz_features), torch.zeros_like(horiz_features)), dim=2)]))

        # Reconstruct vertical component
        rec_vert = ifm((approx_features, [torch.stack((torch.zeros_like(vert_features), vert_features, torch.zeros_like(vert_features)), dim=2)]))

        # Reconstruct diagonal (spatial) component
        rec_diag = ifm((approx_features, [torch.stack((torch.zeros_like(diag_features), torch.zeros_like(diag_features), diag_features), dim=2)]))

        return rec_horiz, rec_vert, rec_diag

#################################################

def swish(x):
    return x * torch.sigmoid(x)


ACT2FN = {"gelu": torch.nn.functional.gelu, "relu": torch.nn.functional.relu, "swish": swish}


class Attention(nn.Module):
    def __init__(self, vis):
        super(Attention, self).__init__()
        self.vis = vis
        self.num_attention_heads = 12
        self.attention_head_size = int(768 / self.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        #self.query = SSL(768)
        self.query = Linear(768, self.all_head_size)
        self.key = Linear(768, self.all_head_size)
        self.value = Linear(768, self.all_head_size)

        self.out = Linear(768, 768)
        self.attn_dropout = Dropout(0.0)
        self.proj_dropout = Dropout(0.1)

        self.softmax = Softmax(dim=-1)

 
    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)
    
    def forward(self, hidden_states):
       
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        
        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_probs = self.softmax(attention_scores)
        weights = attention_probs if self.vis else None
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights

class Mlp(nn.Module):
    def __init__(self):
        super(Mlp, self).__init__()
        self.fc1 = Linear(768, 3072)
        self.fc2 = Linear(3072, 768)
        self.act_fn = ACT2FN["gelu"]
        self.dropout = Dropout(0.1)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.normal_(self.fc1.bias, std=1e-6)
        nn.init.normal_(self.fc2.bias, std=1e-6)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act_fn(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class Embeddings(nn.Module):
    """Construct the embeddings from patch, position embeddings.
    """
    def __init__(self, img_size, in_channels=3):
        super(Embeddings, self).__init__()
        self.num_layers = (3, 4, 9)
        self.width_factor = 1
        img_size = _pair(img_size)

        grid_size = (16,16)
        patch_size = (img_size[0] // 16 // grid_size[0], img_size[1] // 16 // grid_size[1])
        patch_size_real = (patch_size[0] * 16, patch_size[1] * 16)
        n_patches = (img_size[0] // patch_size_real[0]) * (img_size[1] // patch_size_real[1])  
        
        self.hybrid_model = ResNetV2(block_units=self.num_layers, width_factor=self.width_factor)
        in_channels = self.hybrid_model.width * 16
        
        self.patch_embeddings = Conv2d(in_channels=in_channels,
                                       out_channels=768,
                                       kernel_size=patch_size,
                                       stride=patch_size)
        self.position_embeddings = nn.Parameter(torch.zeros(1, n_patches, 768))

        self.dropout = Dropout(0.1)

        self.wavelet = SSL(3) # in_chan= 6 with wavelet
        self.conv_layer = nn.Conv2d(in_channels=6, out_channels=3, kernel_size=1, stride=1, padding=0).to(device)
        self.conv_con = nn.Conv2d(in_channels=1024, out_channels=3, kernel_size=1, stride=1, padding=0).to(device)
    def forward(self, x):
    
        x_h, x_v, x_sp = self.wavelet(x)

        #extract horizontal features by ResNet
        x_h = torch.cat((x, x_h),1)
        x_h = self.conv_layer(x_h)
        x_h, features = self.hybrid_model(x_h)
    
        #extract vertical features by ResNet
        x_v = torch.cat((x, x_v),1)
        x_v = self.conv_layer(x_v)
        x_v, features = self.hybrid_model(x_v)

        #extract diagonal features by ResNet
        x_sp = torch.cat((x, x_sp),1)
        x_sp = self.conv_layer(x_sp)
        x_sp, features = self.hybrid_model(x_sp)

        x = x_h +x_v +x_sp
        
        x = self.patch_embeddings(x)  # (B, hidden. n_patches^(1/2), n_patches^(1/2))
        x = x.flatten(2)
        x = x.transpose(-1, -2)  # (B, n_patches, hidden)

        embeddings = x + self.position_embeddings
        embeddings = self.dropout(embeddings)
        return embeddings, features


class Block(nn.Module):
    def __init__(self, vis):
        super(Block, self).__init__()
        self.hidden_size = 768
        self.attention_norm = LayerNorm(768, eps=1e-6)
        self.ffn_norm = LayerNorm(768, eps=1e-6)
        self.ffn = Mlp()
        self.attn = Attention(vis)

    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x = x + h

        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x = x + h
        return x, weights

class Encoder(nn.Module):
    def __init__(self, vis):
        super(Encoder, self).__init__()
        self.vis = vis
        self.layer = nn.ModuleList()
        self.encoder_norm = LayerNorm(768, eps=1e-6)
        for _ in range(12):  #number of block of tranformers layer
            layer = Block(vis)
            self.layer.append(copy.deepcopy(layer))

    def forward(self, hidden_states):
        attn_weights = []
        for layer_block in self.layer:
            hidden_states, weights = layer_block(hidden_states)
            if self.vis:
                attn_weights.append(weights)
        encoded = self.encoder_norm(hidden_states)
        return encoded, attn_weights


class Transformer(nn.Module):
    def __init__(self, img_size, vis):
        super(Transformer, self).__init__()
        self.embeddings = Embeddings(img_size=img_size)
        self.encoder = Encoder(vis)

    def forward(self, input_ids):
        embedding_output, features = self.embeddings(input_ids)
        encoded, attn_weights = self.encoder(embedding_output)  # (B, n_patch, hidden)
        return encoded, attn_weights, features


class Conv2dReLU(nn.Sequential):
    def __init__(
            self,
            in_channels,
            out_channels,
            kernel_size,
            padding=0,
            stride=1,
            use_batchnorm=True,
    ):
        conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            bias=not (use_batchnorm),
        )
        relu = nn.ReLU(inplace=True)

        bn = nn.BatchNorm2d(out_channels)

        super(Conv2dReLU, self).__init__(conv, bn, relu)


class DecoderBlock(nn.Module):
    def __init__(
            self,
            in_channels,
            out_channels,
            skip_channels=3,
            use_batchnorm=True,
    ):
        super().__init__()
        self.conv1 = Conv2dReLU(
            in_channels + skip_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            use_batchnorm=use_batchnorm,
        )
        self.conv2 = Conv2dReLU(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            use_batchnorm=use_batchnorm,
        )
        self.up = nn.UpsamplingBilinear2d(scale_factor=2)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class SegmentationHead(nn.Sequential):

    def __init__(self, in_channels, out_channels, kernel_size=3, upsampling=1):
        conv2d = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2)
        upsampling = nn.UpsamplingBilinear2d(scale_factor=upsampling) if upsampling > 1 else nn.Identity()
        super().__init__(conv2d, upsampling)


class DecoderCup(nn.Module):
    def __init__(self):
        super().__init__()
        head_channels = 512
        self.conv_more = Conv2dReLU(
            768,
            head_channels,
            kernel_size=3,
            padding=1,
            use_batchnorm=True,
        )

        self.projection_layer = nn.Conv2d(768, 512, kernel_size=1)
        decoder_channels = (256, 128, 64, 16)
        in_channels = [head_channels] + list(decoder_channels[:-1])
        out_channels = decoder_channels
        skip_channels = [512, 256, 64, 16]
        self.n_skip = 3
        for i in range(4-self.n_skip):  # re-select the skip channels according to n_skip
            skip_channels[3-i]=0

        blocks = [
            DecoderBlock(in_ch, out_ch, sk_ch) for in_ch, out_ch, sk_ch in zip(in_channels, out_channels, skip_channels)
        ]
        self.blocks = nn.ModuleList(blocks)

    def forward(self, hidden_states, features=None):
        B, n_patch, hidden = hidden_states.size()  # reshape from (B, n_patch, hidden) to (B, h, w, hidden)
        h, w = int(np.sqrt(n_patch)), int(np.sqrt(n_patch))
        x = hidden_states.permute(0, 2, 1)
        x = x.contiguous().view(B, hidden, h, w)
        
        x = self.conv_more(x)
  
        for i, decoder_block in enumerate(self.blocks):
            if features is not None:
                skip = features[i] if (i < self.n_skip) else None
            else:
                skip = None
            x = decoder_block(x, skip=skip)
        return x

#////////////////////////////////////////////
#Using Transformers as Encoder
#///////////////////////////////////////////
class VisionTransformer(nn.Module):
    def __init__(self, img_size=256, num_classes=21843, zero_head=False, vis=False):
        super(VisionTransformer, self).__init__()
        self.zero_head = zero_head
        self.transformer = Transformer(img_size, vis)
        self.decoder = DecoderCup()
        self.segmentation_head = SegmentationHead(
            in_channels=16,
            out_channels=3,
            kernel_size=3,
        )
        
    def forward(self, x):
        if x.size()[1] == 1:
            x = x.repeat(1,3,1,1)
       
        #x_input = x
        #x = self.wavelet_transform(x)
        #x = x + x_input
        
        x, attn_weights, features = self.transformer(x)  # (B, n_patch, hidden)
        x = self.decoder(x, features)
        
        x = self.segmentation_head(x)
        return x

## Evaluation functions

In [ ]:
# =============================================================================================
# Corrected evaluation for the WavTRGAN revision. Run as ONE cell after cells 1-16 of the notebook
# (so the model classes exist). Before it, run once in its own cell:  !pip install -q lpips torch-fidelity
#
# Part 1  generate_wavtrgan(): runs WavTRGAN on the test pairs WITHOUT random crop / flip and
#         saves input, ground truth and output under the SAME file name (needed for fair,
#         reproducible pairs and for hole-only metrics).
# Part 2  evaluate(): scores every method on the same pairs:
#         - PSNR / SSIM on float images (no uint8 overflow), full image, RGB
#         - PSNR / SSIM on the corrupted region only (hole) and on the rest (known)
#         - LPIPS (AlexNet), FID (torch-fidelity)
#         Files are matched by NAME (sorted), never by glob order.
#
# Folder layout expected for every method: one folder of outputs whose file names are the same
# as in GT_DIR / INPUT_DIR (e.g. 0001.png). Run baselines on INPUT_DIR and save their outputs
# under the same names.
# =============================================================================================
import os, csv
import numpy as np
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

EXTS = (".png", ".jpg", ".jpeg")
HOLE_THRESHOLD = 0.05          # a pixel is "corrupted" if input and ground truth differ by > 0.05 (any channel)


def load_rgb(path, size=None):
    im = Image.open(path).convert("RGB")          # also drops the alpha channel of plt.imsave PNGs
    if size is not None and im.size != size:
        im = im.resize(size, Image.BICUBIC)
    return np.asarray(im, dtype=np.float64)       # float64 in [0, 255]: no uint8 wrap-around


def names_in(folder):
    return sorted(f for f in os.listdir(folder) if f.lower().endswith(EXTS))


# ---------------------------------------------------------------------------------------------
# Part 1: WavTRGAN outputs on fixed pairs (run in the WavTRGAN notebook, after the model cells)
# ---------------------------------------------------------------------------------------------
@torch.no_grad()
def generate_wavtrgan(model, test_root, out_root, direction="b2a", size=256, device="cuda"):
    """test_root/{a,b}: clean / corrupted pairs with identical names. Writes out_root/{input,gt,wavtrgan}."""
    corrupted_dir, clean_dir = (os.path.join(test_root, "b"), os.path.join(test_root, "a")) if direction == "b2a" \
        else (os.path.join(test_root, "a"), os.path.join(test_root, "b"))
    dirs = {k: os.path.join(out_root, k) for k in ("input", "gt", "wavtrgan")}
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    model = model.to(device).eval()
    for name in names_in(corrupted_dir):
        stem = os.path.splitext(name)[0] + ".png"
        inp = load_rgb(os.path.join(corrupted_dir, name), (size, size)) / 255.0
        gt = load_rgb(os.path.join(clean_dir, name), (size, size)) / 255.0
        x = torch.from_numpy(inp).permute(2, 0, 1)[None].float().to(device)   # [0, 1], as in training
        y = model(x).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
        for key, arr in (("input", inp), ("gt", gt), ("wavtrgan", y)):
            Image.fromarray((arr * 255).round().astype(np.uint8)).save(os.path.join(dirs[key], stem))
    print("saved", len(names_in(dirs["input"])), "pairs to", out_root)


# ---------------------------------------------------------------------------------------------
# Part 2: metrics for all methods on the same pairs
# ---------------------------------------------------------------------------------------------
def masked_psnr(gt, pr, m):
    """PSNR over the pixels where m is True (m: H x W bool), images float in [0, 255]."""
    if m.sum() == 0:
        return np.nan
    mse = ((gt - pr) ** 2)[m].mean()
    return np.inf if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)


def masked_ssim(gt, pr, m):
    """Mean of the SSIM map over the pixels where m is True."""
    if m.sum() == 0:
        return np.nan
    _, smap = structural_similarity(gt, pr, data_range=255, channel_axis=2, full=True)
    return smap.mean(axis=2)[m].mean()


def evaluate(methods, gt_dir, input_dir, out_csv="metrics.csv", lpips_on=True, fid_on=True, device=None):
    """methods: {"WavTRGAN": folder, "CAML": folder, ...}. Every folder must contain the same file names."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    gt_names = names_in(gt_dir)
    lp = None
    if lpips_on:
        import lpips
        lp = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
    rows = []
    for method, folder in methods.items():
        stems = {os.path.splitext(f)[0]: f for f in names_in(folder)}
        pairs = [(g, stems[os.path.splitext(g)[0]]) for g in gt_names if os.path.splitext(g)[0] in stems]
        missing = len(gt_names) - len(pairs)
        assert pairs, "%s: no file names in common with %s" % (method, gt_dir)
        r = {k: [] for k in ("psnr", "ssim", "psnr_hole", "ssim_hole", "psnr_known", "lpips", "ratio")}
        for g, f in pairs:
            gt = load_rgb(os.path.join(gt_dir, g))
            pr = load_rgb(os.path.join(folder, f), size=(gt.shape[1], gt.shape[0]))
            inp = load_rgb(os.path.join(input_dir, g), size=(gt.shape[1], gt.shape[0]))
            hole = (np.abs(inp - gt).max(axis=2) > HOLE_THRESHOLD * 255)
            r["psnr"].append(peak_signal_noise_ratio(gt, pr, data_range=255))
            r["ssim"].append(structural_similarity(gt, pr, data_range=255, channel_axis=2))
            r["psnr_hole"].append(masked_psnr(gt, pr, hole))
            r["ssim_hole"].append(masked_ssim(gt, pr, hole))
            r["psnr_known"].append(masked_psnr(gt, pr, ~hole))
            r["ratio"].append(hole.mean())
            if lp is not None:
                t = lambda a: torch.from_numpy(a / 127.5 - 1).permute(2, 0, 1)[None].float().to(device)
                with torch.no_grad():
                    r["lpips"].append(lp(t(pr), t(gt)).item())
        finite = lambda v: np.array(v, dtype=float)[np.isfinite(v)]
        row = {"method": method, "images": len(pairs), "missing": missing,
               "PSNR": finite(r["psnr"]).mean(), "SSIM": np.mean(r["ssim"]),
               "PSNR_hole": finite(r["psnr_hole"]).mean(), "SSIM_hole": np.nanmean(r["ssim_hole"]),
               "PSNR_known": finite(r["psnr_known"]).mean(),
               "LPIPS": np.mean(r["lpips"]) if r["lpips"] else np.nan,
               "hole_ratio": np.mean(r["ratio"])}
        if fid_on:
            import torch_fidelity
            row["FID"] = torch_fidelity.calculate_metrics(
                input1=folder, input2=gt_dir, cuda=torch.cuda.is_available(), fid=True, verbose=False
            )["frechet_inception_distance"]
        rows.append(row)
        print("%-22s n=%d%s  PSNR %.2f  SSIM %.4f | hole: PSNR %.2f SSIM %.4f | known PSNR %.2f | LPIPS %.4f%s" % (
            method, row["images"], " (%d missing!)" % missing if missing else "", row["PSNR"], row["SSIM"],
            row["PSNR_hole"], row["SSIM_hole"], row["PSNR_known"], row["LPIPS"],
            "  FID %.2f" % row["FID"] if "FID" in row else ""))
    with open(out_csv, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print("mean corrupted fraction of the test set: %.1f%%  -> %s" % (100 * rows[0]["hole_ratio"], out_csv))
    return rows


# ---------------------------------------------------------------------------------------------
# Sanity check: the old uint8 code vs this code on two unrelated images (should be ~8 dB here)
# ---------------------------------------------------------------------------------------------
def sanity_check():
    rng = np.random.default_rng(0)
    a = rng.integers(0, 256, (64, 64, 3)).astype(np.uint8)
    b = rng.integers(0, 256, (64, 64, 3)).astype(np.uint8)
    old = 20 * np.log10(255 / np.sqrt(np.mean((a - b) ** 2)))           # the old, wrong formula
    new = peak_signal_noise_ratio(a.astype(np.float64), b.astype(np.float64), data_range=255)
    print("unrelated random images: old uint8 PSNR %.2f dB (wrong), corrected %.2f dB" % (old, new))

## Configuration and run

In [ ]:
# =============================================================================================
# CONFIGURATION (edit these paths)
# =============================================================================================
MODEL_PATH = "/kaggle/input/datasets/zelligproject/netg-withwav-40/netG_model_epoch_40.pth"  # model used in the paper
TEST_ROOT  = "/kaggle/input/celeba-hq-img-full/CelebA-HQ-img/test"   # contains a/ (clean) and b/ (corrupted)
OUT_ROOT   = "/kaggle/working/eval"                                # input/, gt/, wavtrgan/ are written here
BASELINES  = {                                                     # optional: folders of baseline outputs
    # "CAML": "/kaggle/working/eval/caml",                         # (same file names as OUT_ROOT/input)
}

sanity_check()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.load(MODEL_PATH, map_location=device, weights_only=False)
if isinstance(model, dict):                                        # weights only: build the model first
    raise RuntimeError("MODEL_PATH holds a state_dict: create the VisionTransformer and load_state_dict it here")
if isinstance(model, torch.nn.DataParallel):
    model = model.module
print("parameters: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))

generate_wavtrgan(model, TEST_ROOT, OUT_ROOT, direction="b2a", size=256, device=device)
methods = {"WavTRGAN": os.path.join(OUT_ROOT, "wavtrgan"), **BASELINES}
results = evaluate(methods, gt_dir=os.path.join(OUT_ROOT, "gt"), input_dir=os.path.join(OUT_ROOT, "input"),
                   out_csv="/kaggle/working/metrics.csv", lpips_on=True, fid_on=True, device=device)

## Visual check

In [ ]:
# Visual check: input | WavTRGAN | ground truth for a few test images
import matplotlib.pyplot as plt
names = names_in(os.path.join(OUT_ROOT, "gt"))[:4]
fig, ax = plt.subplots(len(names), 3, figsize=(9, 3 * len(names)), squeeze=False)
for r, n in enumerate(names):
    for c, k in enumerate(("input", "wavtrgan", "gt")):
        ax[r, c].imshow(Image.open(os.path.join(OUT_ROOT, k, n)))
        ax[r, c].set_title(k if r == 0 else "")
        ax[r, c].axis("off")
plt.tight_layout()
plt.show()